# Support Odata requests in the catalog

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-1223

The goal of this story is to manage odata requests through the catalog for PRIP, and return an Odata response, with a filter on publication date. This demo requests products using both the stac (usual) and odata (new) endpoint and compares the numbers of stac assets and products received in response.

In [ ]:
import os
import requests
# Init environment before running a demo notebook.
from resources.utils import *

init_demo()
# Reload the global vars again
from resources.utils import *

In [ ]:
# Define 2 dates to filter data on
d1 = "2026-06-01T00:00:00Z"
d2 = "2026-07-01T00:00:00Z"

# Do we want to run a staging before sending the request? If no, make sure there is already data in the catalog
perform_staging = False

## Step 1: stage test data

On a cluster with already test data staged in the catalog, this test is optional because the Odata query is on all the data in the catalog and not only the one in a specific collection so it usually find matches without the need of more data.   
On an empty catalog, this part stages two files: one that is inside the time interval we test, one that is outside, so we can make sure the Odata request returns only the file that is inside the interval and not the other one.

In [ ]:
if perform_staging:
    from resources.widget_utils import *
    from rs_workflows.aux_flow import aux_staging
    from resources.dask_clusters.dask_main_env import *
    
    # Create test collection
    CATALOG_COLLECTION_ID = "odata_demo_collection"
    collection = create_test_collection(CATALOG_COLLECTION_ID)
    
    # Create staging cluster
    await init_dask_cluster_staging()
    
    # Deploy the Prefect flows
    s3_code_folder = f"users/{OWNER_ID}/code"
    dpr_processing_deploy, aux_staging_deploy, cadip_staging_deploy = await deploy_prefect(
        deploy_file="../sprint38/dpr_processing_flow/dpr_processing_flow.yaml", 
        s3_code_folder=s3_code_folder, 
        work_pool_name=os.environ["PREFECT_WORK_POOL_SANDBOX"]
    )
    
    # Prefect flow environment arguments
    flow_env_args = {
      "env": {
        "owner_id": OWNER_ID,
        "logging_level": prefect_logging_level_radio.value,
      },
    }

In [ ]:
if perform_staging:
    # Auxip staging
    ids = [
        "S00__ADF_ECMWF_20260608T210000_20260609T150000_20260611T123606.zarr",
        "S1A_OPER_AUX_OBMEMC_PDMC_20240106T000000"
    ]
    id_filter = ','.join([f"'{id}'" for id in ids])
    id_filter = f"id in ({id_filter})"
    cql2_filter = {"filter": id_filter}
    params = {
        **flow_env_args,
        "cql2_filter": cql2_filter,
        "catalog_collection_identifier": CATALOG_COLLECTION_ID,
    }
    await run_prefect(
        deploy_name=aux_staging_deploy, 
        py_func=aux_staging, 
        params=params
    )
    
    # Check staged items in the catalog
    staged = catalog_client.search(
        owner_id=OWNER_ID, 
        collections=[CATALOG_COLLECTION_ID]
    ).items
    display(staged)
    assert len(staged) == 2
    for item in staged:
        assert (ret := item.id) in ids, f"{ret} not in {ids}"

### Step 2: Odata request

No API endpoint was created yet, so we send the request directly.

In [ ]:
openapi_endpoint = f"/catalog/odata/v1/Products?$filter=PublicationDate gt {d1} and PublicationDate lt {d2}"
url = f"{catalog_client.href_service}{openapi_endpoint}"
headers = {'Accept': 'application/json',
          "X-API-Key": os.getenv("RSPY_APIKEY")}

openapi_response = requests.get(url, headers=headers)
odata_items = openapi_response.json()
display(odata_items)

### Step 3: equivalent STAC request

In [ ]:
stac_filter = {
    "filter": {
        "op": "and",
        "args": [
            {"op": ">=", "args": [{"property": "published"}, "2026-06-01T00:00:00Z"]},
            {"op": "<=", "args": [{"property": "published"}, "2026-06-30T00:00:00Z"]}
        ]
    },
    "filter-lang": "cql2-json"
}

stac_items = catalog_client.search(stac_filter=stac_filter)
display(stac_items)

### Step 4: check the filter

In [ ]:
[f["PublicationDate"] for f in odata_items]

### Step 5: compare stac and odata responses

We don't compare directly the number of returned items: we compare the number of stac assets (from the stac request) against the numbers of returned products (with the odata request).

In [ ]:
len(odata_items)

In [ ]:
# On the dev cluster, some features don't have assets
n_stac_assets = sum([len(f.assets) for f in stac_items if hasattr(f, "assets")])
n_stac_assets